# Day 09: 强化学习基础 -- 从经典 RL 到 LLM 对齐

**目标：** 理解强化学习核心概念，以及它们如何映射到大语言模型训练中。

**学习路线：**
1. RL 的本质：Agent-环境交互循环
2. RL 概念到 LLM 的映射
3. REINFORCE 算法：最基础的策略梯度
4. 从 REINFORCE 到 PPO 的演进
5. GRPO：DeepSeek-R1 的秘密武器

> **前置知识：** 了解 LLM 基础（tokenizer、next-token prediction）

---
## 1. RL 的本质：Agent-环境交互循环

强化学习的核心是一个 **循环过程**：

```
Agent（智能体）  <---- 观测 + 奖励 ----  环境
     |                                    ^
     +------------ 动作 -----------------+
```

- **Agent** 根据当前观测选择动作
- **环境** 根据动作返回新的观测和奖励
- Agent 的目标：**最大化累计奖励**

### RL vs 监督学习：根本区别

| 维度 | 监督学习 (SFT) | 强化学习 (RL) |
|------|---------------|---------------|
| **数据** | 固定的 (x, y) 数据集 | Agent 自己生成数据 |
| **反馈** | 正确答案（标签） | 标量奖励信号 |
| **优化** | 最小化与标签的差距 | 最大化期望累计奖励 |
| **探索** | 不需要 | 需要探索-利用平衡 |
| **延迟** | 即时反馈 | 可能延迟反馈 |

> SFT 告诉模型"正确答案长什么样"，RL 告诉模型"什么样的回答更受欢迎"。

In [ ]:
# 环境准备
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "day09_rl_basics" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

np.random.seed(42)
print("环境准备完成!")

### Agent-环境循环可视化

In [ ]:
# RL 循环示例
print("=" * 50)
print("RL 循环示例：LLM 回复评分")
print("=" * 50)

prompt = "请解释什么是机器学习"
responses = [
    ("机器学习就是让计算机从数据中学习规律。", 0.6),
    ("机器学习是AI的子领域，通过算法让机器从数据中学习模式和规律。", 0.9),
    ("呃...就是...那个...机器...学习...", 0.1),
]

print(f"\n[Prompt/State]: {prompt}")
print("\n[Agent 采样多个回复 (Actions)]:")
for i, (resp, reward) in enumerate(responses):
    tag = "+++" if reward > 0.7 else "o" if reward > 0.4 else "---"
    print(f'  回复 {i+1}: "{resp}"')
    print(f"         奖励: {reward} [{tag}]")

print("\n[策略更新]:")
print("  -> 提高回复2的概率（高奖励），降低回复3的概率（低奖励）")
print("  这就是 REINFORCE 算法的核心思想!")

---
## 2. RL 到 LLM 概念映射

| 经典 RL 概念 | LLM 对齐中的对应 | 说明 |
|:------------|:----------------|:-----|
| **State** (状态) | 已生成的 token 序列 | 每生成一个 token，状态变化 |
| **Action** (动作) | 选择下一个 token | 词表大小 = 动作空间 |
| **Policy** | 语言模型本身 | 模型参数 = 策略参数 |
| **Reward** (奖励) | 人类偏好评分 / RM 输出 | 回复结束后给出 |
| **Episode** (回合) | 生成一个完整回复 | prompt 到 EOS |
| **Trajectory** (轨迹) | prompt->t1->t2->...->EOS | 整个 token 序列 |
| **Value** V(s) | 期望回报 | Critic 模型估计 |

In [ ]:
# 可视化概念映射
fig, ax = plt.subplots(figsize=(12, 7))
ax.axis("off")
ax.set_title("RL <-> LLM 概念映射", fontsize=16, fontweight="bold", pad=20)

mapping = [
    ("经典 RL 概念", "LLM 对齐中的对应"),
    ("State (状态)", "已生成的 token 序列"),
    ("Action (动作)", "选择下一个 token"),
    ("Policy", "语言模型 LM(token|context)"),
    ("Reward R", "人类偏好评分 / RM 输出"),
    ("Episode", "生成一个完整回复"),
    ("Trajectory", "prompt->t1->t2->...->EOS"),
    ("Value V(s)", "从当前位置的期望回报"),
    ("REINFORCE", "基础策略梯度"),
    ("PPO", "RLHF 中的策略优化"),
    ("GRPO", "无 Critic 的组内排名"),
]
text = "\n".join(f"  {left:<22s} <->  {right}" for left, right in mapping)
ax.text(0.1, 0.9, text, transform=ax.transAxes, fontsize=11,
        verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.9))
plt.tight_layout()
plt.show()

---
## 3. REINFORCE 算法

### 3.1 核心思想

REINFORCE 核心公式：**grad J = E[ sum_t  grad log pi(a_t|s_t) * G_t ]**

**直觉**：
- 高回报 G_t -> 提高这些动作的概率
- 低回报 G_t -> 降低这些动作的概率
- log pi(a_t|s_t) 就是 LLM 对 next token 的对数概率

### 3.2 在 LLM 中
- theta = 模型参数，a = 生成的 token，R = 奖励模型评分

In [ ]:
# 多臂赌博机环境
class ToyBanditEnv:
    def __init__(self, n_arms=4):
        self.n_arms = n_arms
        self.true_means = [0.5, 0.8, 0.2, 0.6]  # 人类偏好
        self.true_stds = [0.1, 0.15, 0.1, 0.4]
        self.labels = ["简短直接", "详细解释", "废话连篇", "创意发散"]

    def step(self, action):
        reward = np.random.normal(self.true_means[action], self.true_stds[action])
        return np.clip(reward, 0, 1)

env = ToyBanditEnv()
print("多臂赌博机环境 (类比LLM回复风格):")
for i in range(env.n_arms):
    print(f"  臂 {i} ({env.labels[i]}): 均值={env.true_means[i]}, 标准差={env.true_stds[i]}")

In [ ]:
# REINFORCE Agent
class REINFORCEAgent:
    def __init__(self, n_actions, lr=0.1, baseline_decay=0.9):
        self.theta = np.zeros(n_actions)
        self.lr = lr
        self.baseline = 0.0
        self.baseline_decay = baseline_decay

    def get_policy(self):
        exp_theta = np.exp(self.theta - np.max(self.theta))
        return exp_theta / exp_theta.sum()

    def select_action(self):
        probs = self.get_policy()
        return np.random.choice(len(probs), p=probs)

    def update(self, action, reward):
        advantage = reward - self.baseline
        self.baseline = self.baseline_decay * self.baseline + (1 - self.baseline_decay) * reward
        probs = self.get_policy()
        grad_log_pi = np.zeros_like(self.theta)
        grad_log_pi[action] = 1.0
        grad_log_pi -= probs
        self.theta += self.lr * advantage * grad_log_pi

print("REINFORCEAgent 定义完成")

In [ ]:
# 运行 REINFORCE 实验
env = ToyBanditEnv()
agent = REINFORCEAgent(n_actions=env.n_arms, lr=0.15)
n_episodes = 500
rewards_history, policy_history, action_history = [], [], []

for episode in range(n_episodes):
    action = agent.select_action()
    reward = env.step(action)
    agent.update(action, reward)
    rewards_history.append(reward)
    policy_history.append(agent.get_policy().copy())
    action_history.append(action)
    if episode % 100 == 0:
        avg_r = np.mean(rewards_history[-50:]) if len(rewards_history) >= 50 else np.mean(rewards_history)
        p = agent.get_policy()
        print(f"Episode {episode:4d} | 平均奖励: {avg_r:.3f} | 策略: [{', '.join(f'{x:.2f}' for x in p)}]")

final_policy = agent.get_policy()
best = np.argmax(final_policy)
print(f"\n最终策略: [{', '.join(f'{x:.3f}' for x in final_policy)}]")
print(f"最优动作: {env.labels[best]} (概率 {final_policy[best]:.3f})")
print(f"真实最优: {env.labels[np.argmax(env.true_means)]} (均值 {max(env.true_means):.3f})")

In [ ]:
# 可视化训练过程
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 图1: 奖励曲线
ax = axes[0, 0]
window = 20
rewards_arr = np.array(rewards_history)
smoothed = np.convolve(rewards_arr, np.ones(window)/window, mode="valid")
ax.plot(smoothed, "b-", alpha=0.8, linewidth=1.5)
ax.axhline(y=max(env.true_means), color="r", linestyle="--", alpha=0.5, label=f"最优 ({max(env.true_means):.2f})")
ax.set_title("REINFORCE 奖励曲线"); ax.set_xlabel("Episode"); ax.set_ylabel("奖励"); ax.legend(); ax.grid(True, alpha=0.3)

# 图2: 策略演变
ax = axes[0, 1]
policies_arr = np.array(policy_history)
for i in range(env.n_arms):
    ax.plot(policies_arr[:, i], label=env.labels[i], linewidth=1.5)
ax.set_title("策略演变"); ax.set_xlabel("Episode"); ax.set_ylabel("概率"); ax.legend(); ax.grid(True, alpha=0.3)

# 图3: 策略梯度直觉
ax = axes[1, 0]
tokens = ["好", "一般", "差"]
x_pos = np.arange(3); w = 0.35
ax.bar(x_pos - w/2, [1/3]*3, w, label="更新前", color="#FF6B6B", alpha=0.8)
ax.bar(x_pos + w/2, [0.6, 0.3, 0.1], w, label="更新后", color="#4ECDC4", alpha=0.8)
ax.set_xticks(x_pos); ax.set_xticklabels(tokens)
ax.set_title("策略梯度直觉"); ax.set_ylabel("概率"); ax.legend(); ax.grid(True, alpha=0.3, axis="y")

# 图4: 动作频率
ax = axes[1, 1]
early = action_history[:100]; late = action_history[-100:]
x_pos = np.arange(env.n_arms)
ax.bar(x_pos - w/2, [early.count(i)/100 for i in range(env.n_arms)], w, label="前100次", color="#FF6B6B", alpha=0.8)
ax.bar(x_pos + w/2, [late.count(i)/100 for i in range(env.n_arms)], w, label="后100次", color="#4ECDC4", alpha=0.8)
ax.set_xticks(x_pos); ax.set_xticklabels(env.labels)
ax.set_title("探索到利用"); ax.set_ylabel("频率"); ax.legend(); ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout(); plt.show()

### 3.3 REINFORCE 的问题

REINFORCE 虽然简洁，但有严重问题：**高方差**

- 每次采样的回报 G_t 波动很大，梯度估计不稳定
- 训练需要大量样本才能收敛

演进路线：`REINFORCE -> + Baseline -> Actor-Critic -> PPO -> GRPO`

---
## 4. 从 REINFORCE 到 PPO 的演进

### 4.1 Baseline 减少方差
引入基线 b：**grad J = E[ grad log pi(a|s) * (G_t - b) ]**，常见选择 b = V(s)

### 4.2 Actor-Critic
- **Actor**：策略网络，决定动作
- **Critic**：价值网络 V，估计状态价值
- 优势：A(s, a) = R + gamma * V(s') - V(s)

### 4.3 PPO
在 Actor-Critic 上加 **clipping**，限制更新幅度：
**L = E[ min( r * A, clip(r, 1-eps, 1+eps) * A ) ]**
其中 r = pi_new / pi_old

In [ ]:
# PPO Clipping 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
epsilon = 0.2
r = np.linspace(0.5, 1.5, 200)
A_pos = 1.0
unclipped = r * A_pos
clipped = np.clip(r, 1-epsilon, 1+epsilon) * A_pos
objective = np.minimum(unclipped, clipped)
ax.plot(r, unclipped, "b--", alpha=0.5, label="未裁剪 r*A")
ax.plot(r, clipped, "r--", alpha=0.5, label="裁剪 clip(r)*A")
ax.plot(r, objective, "g-", linewidth=2, label="PPO min(...)")
ax.axvline(x=1-epsilon, color="gray", linestyle=":", alpha=0.5)
ax.axvline(x=1+epsilon, color="gray", linestyle=":", alpha=0.5)
ax.set_title(f"PPO Clipping (eps={epsilon})"); ax.set_xlabel("概率比 r"); ax.set_ylabel("目标"); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.axis("off")
ax.set_title("演进路线", fontsize=13, fontweight="bold")
txt = ("REINFORCE (基础)\n  高方差\n  |\n  v\n"
       "+ Baseline\n  减方差\n  |\n  v\n"
       "Actor-Critic\n  Actor + Critic\n  |\n  v\n"
       "PPO\n  + Clipping\n  InstructGPT/ChatGPT\n  |\n  v\n"
       "GRPO\n  无 Critic\n  组内归一化\n  DeepSeek-R1")
ax.text(0.1, 0.95, txt, transform=ax.transAxes, fontsize=10,
        verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="lightcyan", alpha=0.9))
plt.tight_layout(); plt.show()

In [ ]:
# 有无 baseline 对比
def run_experiments(n_runs=20, n_episodes=300):
    results = {"with_baseline": [], "without_baseline": []}
    for _ in range(n_runs):
        for use_bl in [True, False]:
            e = ToyBanditEnv(); ag = REINFORCEAgent(n_actions=4, lr=0.15)
            if not use_bl: ag.baseline_decay = 0; ag.baseline = 0
            rews = []
            for ep in range(n_episodes):
                a = ag.select_action(); r = e.step(a)
                if use_bl:
                    ag.update(a, r)
                else:
                    probs = ag.get_policy(); g = np.zeros_like(ag.theta); g[a] = 1.0; g -= probs
                    ag.theta += ag.lr * r * g
                rews.append(r)
            key = "with_baseline" if use_bl else "without_baseline"
            results[key].append(np.mean(rews[-50:]))
    return results

res = run_experiments()
print(f"有 Baseline: 均值={np.mean(res['with_baseline']):.3f}, 标准差={np.std(res['with_baseline']):.3f}")
print(f"无 Baseline: 均值={np.mean(res['without_baseline']):.3f}, 标准差={np.std(res['without_baseline']):.3f}")

---
## 5. GRPO：不需要 Critic 的策略优化

### 核心思想
GRPO 用 **组内相对排名** 代替 Critic 模型。

### 算法步骤
对每个 prompt x：
1. **采样** G 个回复
2. **计算奖励** r_1, ..., r_G
3. **组内归一化**：A_i = (r_i - mean(r)) / std(r)
4. **策略梯度更新**（带 clipping 和 KL 惩罚）

### 对比

| 特性 | PPO | GRPO |
|------|-----|------|
| Critic 模型 | 需要 | 不需要 |
| 优势估计 | GAE | 组内归一化 |
| 显存 | 2x | 1x |
| 适用 | 通用 | 可验证任务 |
| 代表 | InstructGPT | DeepSeek-R1 |

In [ ]:
# GRPO 组内归一化示例
print("=" * 50)
print("GRPO 组内归一化示例")
print("=" * 50)

G = 6
np.random.seed(42)
rewards = np.random.uniform(0, 1, G)
mean_r = np.mean(rewards); std_r = np.std(rewards)
advantages = (rewards - mean_r) / (std_r + 1e-8)

print(f"\n采样 {G} 个回复的奖励: {[f'{r:.3f}' for r in rewards]}")
print(f"均值: {mean_r:.3f}, 标准差: {std_r:.3f}")
print("\n归一化优势:")
for i in range(G):
    action = "增加" if advantages[i] > 0 else "降低"
    print(f"  回复 {i}: r={rewards[i]:.3f} -> A={advantages[i]:+.3f} ({action}概率)")

In [ ]:
# 可视化 GRPO
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors = ["#4ECDC4" if a > 0 else "#FF6B6B" for a in advantages]
ax.bar(range(G), advantages, color=colors, alpha=0.8)
ax.axhline(y=0, color="black", linewidth=0.5)
for i in range(G):
    ax.text(i, advantages[i] + 0.1*np.sign(advantages[i]), f"r={rewards[i]:.2f}", ha="center", fontsize=9)
ax.set_title("GRPO 组内归一化优势"); ax.set_xlabel("回复编号"); ax.set_ylabel("优势 A_hat"); ax.grid(True, alpha=0.3, axis="y")

ax = axes[1]
ax.axis("off"); ax.set_title("GRPO vs PPO", fontsize=13, fontweight="bold")
t = ("PPO:                    GRPO:\n"
     "- 需要 Critic            - 不需要 Critic\n"
     "- 单样本优势              - 组内归一化\n"
     "- 内存 2x               - 内存 1x\n"
     "- 通用                   - 可验证任务\n"
     "- InstructGPT            - DeepSeek-R1")
ax.text(0.05, 0.9, t, transform=ax.transAxes, fontsize=10,
        verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.9))
plt.tight_layout(); plt.show()

In [ ]:
# 模拟 GRPO 训练
print("模拟 GRPO 训练 (多臂赌博机)")
env_grpo = ToyBanditEnv()
theta_grpo = np.zeros(env_grpo.n_arms)
G, lr, n_rounds = 4, 0.2, 200
grpo_rewards, grpo_policies = [], []

for ri in range(n_rounds):
    exp_t = np.exp(theta_grpo - np.max(theta_grpo)); probs = exp_t / exp_t.sum()
    actions = np.random.choice(len(probs), size=G, p=probs)
    rews = np.array([env_grpo.step(a) for a in actions])
    m, s = np.mean(rews), np.std(rews) + 1e-8
    advs = (rews - m) / s
    for a, adv in zip(actions, advs):
        g = np.zeros_like(theta_grpo); g[a] = 1.0; g -= probs
        theta_grpo += lr / G * adv * g
    grpo_rewards.append(np.mean(rews)); grpo_policies.append(probs.copy())
    if ri % 50 == 0:
        print(f"Round {ri:3d} | 奖励: {np.mean(grpo_rewards[-20:]):.3f} | 策略: [{', '.join(f'{p:.2f}' for p in probs)}]")

fp = np.exp(theta_grpo - np.max(theta_grpo)); fp = fp / fp.sum()
print(f"\n最终策略: [{', '.join(f'{p:.3f}' for p in fp)}]")

In [ ]:
# GRPO 训练可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
w = 10; ga = np.array(grpo_rewards)
sm = np.convolve(ga, np.ones(w)/w, mode="valid")
ax.plot(sm, "b-", linewidth=1.5)
ax.axhline(y=max(env_grpo.true_means), color="r", linestyle="--", alpha=0.5)
ax.set_title("GRPO 奖励曲线"); ax.set_xlabel("Round"); ax.set_ylabel("奖励"); ax.grid(True, alpha=0.3)

ax = axes[1]
pg = np.array(grpo_policies)
for i in range(env_grpo.n_arms): ax.plot(pg[:, i], label=env_grpo.labels[i], linewidth=1.5)
ax.set_title("GRPO 策略演变"); ax.set_xlabel("Round"); ax.set_ylabel("概率"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 6. 总结

### 核心要点

1. **RL 的本质** 是 Agent 通过与环境交互学习最优策略
2. **RL 到 LLM 的映射** 是理解 RLHF/GRPO 的基础
3. **REINFORCE** 是最基础的策略梯度，但方差高
4. **PPO** 通过 Critic + Clipping 解决稳定性问题
5. **GRPO** 用组内归一化代替 Critic，更简洁高效

### 演进路线
```
REINFORCE -> + Baseline -> Actor-Critic -> PPO (InstructGPT) -> GRPO (DeepSeek-R1)
```

| 方法 | 优势估计 | 需要 Critic | 代表 |
|------|---------|------------|------|
| REINFORCE | 原始回报 | 否 | 基础研究 |
| Actor-Critic | TD error | 是 | A2C/A3C |
| PPO | GAE | 是 | InstructGPT |
| GRPO | 组内归一化 | **否** | DeepSeek-R1 |

### 思考题

1. **为什么 GRPO 特别适合有明确奖励函数的任务（如数学、代码）？**

2. **GRPO 组大小 G 太小(G=2)或太大(G=64)分别有什么问题？**

3. **为什么需要 KL 散度惩罚？去掉会怎样？**

4. **REINFORCE 的 baseline 和 GRPO 的组内均值有什么相似之处？**

In [ ]:
# 附加：组大小 G 对方差的影响
print("组大小 G 对 GRPO 优势估计方差的影响")
G_values = [2, 4, 8, 16, 32]
variances = []
for Gv in G_values:
    vs = []
    for _ in range(100):
        r = np.random.uniform(0, 1, Gv)
        a = (r - np.mean(r)) / (np.std(r) + 1e-8)
        vs.append(np.var(a))
    variances.append(np.mean(vs))
    print(f"  G={Gv:2d} | 优势方差: {np.mean(vs):.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(G_values, variances, "b-o", linewidth=2, markersize=8)
ax.set_title("组大小 G 对优势估计方差的影响"); ax.set_xlabel("G"); ax.set_ylabel("方差"); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()